# 🔍 Clase 3 — Valores Faltantes: Detección, Diagnóstico e Imputación (R)

## Aplicaciones de Minería de Datos a Economía y Finanzas
### Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026

**Docente:** Dr. Darío Ezequiel Díaz · drdarioezequieldiaz@gmail.com

**Fecha:** Jueves 23 de abril de 2026

---

### Objetivos del notebook

1. **Simular** los tres mecanismos de Rubin (MCAR, MAR, MNAR) sobre un dataset financiero real.
2. **Visualizar** patrones de ausencia con `naniar`, `VIM` y shadow plots.
3. **Diagnosticar** el mecanismo mediante test de Little, regresión logística del indicador de ausencia y correlación point-biserial.
4. **Imputar** con cinco estrategias: eliminación, media/mediana, `mice` (PMM), `mice` (CART) y `missForest`.
5. **Comparar** el impacto de cada estrategia sobre la distribución y la capacidad predictiva.
6. **Demostrar** empíricamente el sesgo de la *listwise deletion* bajo MAR/MNAR.

### Estrategia pedagógica

Partimos del **German Credit Dataset completo** (0% missing). Introducimos valores faltantes artificialmente bajo cada mecanismo, lo que permite **comparar cada imputación contra el valor real** — algo imposible con datos faltantes genuinos.

> **Referencia teórica:** Apunte de Clase 2, Parte I — *Valores Faltantes en Bases de Datos Financieras* (Díaz, 2026).

---

### ⚠️ Instrucciones de configuración en Google Colab

**Paso 1:** Ejecutar la siguiente celda con el runtime en **Python** para montar Google Drive.

**Paso 2:** Ir a *Entorno de ejecución → Cambiar tipo de entorno de ejecución → R*.

**Paso 3:** Continuar ejecutando desde la Sección 1 en adelante.


---
## 0. Montar Google Drive (ejecutar en runtime Python)

> **⚠️ Esta celda debe ejecutarse con el runtime en Python.** Una vez montado el Drive, cambiar a runtime R y continuar desde la Sección 1.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 0. MONTAR GOOGLE DRIVE (ejecutar en runtime Python)
# ══════════════════════════════════════════════════════════════
# Después de ejecutar esta celda, ir a:
#   Entorno de ejecución → Cambiar tipo de entorno de ejecución → R

from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive montado en /content/drive")
print("   Ahora cambiar el runtime a R y continuar desde la Sección 1.")


---
## 1. Configuración del entorno R

Cargamos el entorno de paquetes desde el script de configuración almacenado en Google Drive. Luego verificamos e instalamos los paquetes específicos que este notebook requiere y que podrían no estar incluidos en el setup base.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 1. CONFIGURACIÓN DEL ENTORNO R
# ══════════════════════════════════════════════════════════════

# Cargar el entorno base desde Drive
source("/content/drive/MyDrive/R_Colab/setup_R_colab.R")

# Paquetes específicos requeridos por este notebook
paquetes_requeridos <- c("tidyverse", "naniar", "VIM", "mice",
                         "corrplot", "gridExtra", "pROC", "missForest")

# Verificar e instalar solo los que falten post-setup
faltantes <- paquetes_requeridos[!(paquetes_requeridos %in% installed.packages()[, "Package"])]
if (length(faltantes) > 0) {
  cat(sprintf("📦 Instalando %d paquete(s) no incluido(s) en el setup: %s\n",
              length(faltantes), paste(faltantes, collapse = ", ")))
  install.packages(faltantes, quiet = TRUE)
} else {
  cat("✅ Todos los paquetes requeridos ya están disponibles.\n")
}

# Carga de librerías
suppressPackageStartupMessages({
  library(tidyverse)
  library(naniar)
  library(VIM)
  library(mice)
  library(corrplot)
  library(gridExtra)
  library(pROC)
})

# Intentar cargar missForest (puede fallar si no se instala)
miss_forest_ok <- tryCatch({
  library(missForest)
  TRUE
}, error = function(e) {
  cat("ℹ️  missForest no disponible. Se usarán los otros métodos.\n")
  FALSE
})

# Configuración estética global
theme_set(theme_minimal(base_size = 12, base_family = "sans") +
          theme(plot.title = element_text(face = "bold", size = 14),
                plot.subtitle = element_text(color = "gray40"),
                legend.position = "bottom"))

# Paleta de colores del curso
col_good  <- "#2a9d8f"
col_bad   <- "#e76f51"
col_acc   <- "#f4a261"
palette_curso <- c("good" = col_good, "bad" = col_bad)

cat("\n╔══════════════════════════════════════════════════╗\n")
cat("║   Entorno R configurado correctamente            ║\n")
cat("╠══════════════════════════════════════════════════╣\n")
cat(sprintf("║   R          : %-37s║\n", R.version$version.string))
cat(sprintf("║   tidyverse  : %-37s║\n", packageVersion("tidyverse")))
cat(sprintf("║   mice       : %-37s║\n", packageVersion("mice")))
cat(sprintf("║   naniar     : %-37s║\n", packageVersion("naniar")))
cat(sprintf("║   VIM        : %-37s║\n", packageVersion("VIM")))
cat(sprintf("║   pROC       : %-37s║\n", packageVersion("pROC")))
cat("╚══════════════════════════════════════════════════╝\n")


### 📝 Interpretación

El entorno se configura en dos etapas: primero el script base de Drive carga los ~237 paquetes precompilados, y luego verificamos los paquetes específicos de este notebook. Este patrón evita recompilar paquetes que ya existen y solo instala los estrictamente necesarios.

Los paquetes clave para el tratamiento de valores faltantes en R:
- **`naniar`**: Visualización y diagnóstico (Tierney & Cook, 2023). Incluye `vis_miss()`, `gg_miss_var()`, `mcar_test()` y los *shadow plots*.
- **`mice`**: Imputación múltiple por ecuaciones encadenadas (Van Buuren & Groothuis-Oudshoorn, 2011). Métodos PMM, CART, norm, rf.
- **`VIM`**: Visualización avanzada de patrones de ausencia (Kowarik & Templ, 2016).
- **`pROC`**: AUC-ROC para el diagnóstico del mecanismo de ausencia.

---
## 2. Carga del dataset y línea base

Cargamos el German Credit Dataset completo. Al no tener valores faltantes, nos sirve como **ground truth** para evaluar la calidad de cada imputación.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. CARGA DEL DATASET
# ══════════════════════════════════════════════════════════════

# Descarga desde OpenML (alternativa: cargar desde Drive)
url_data <- "https://www.openml.org/data/get_csv/31/dataset_31_credit-g.arff"

df_original <- tryCatch({
  df <- read_csv(url_data, show_col_types = FALSE)
  cat("✅ Dataset descargado desde OpenML\n")
  df
}, error = function(e) {
  cat("ℹ️  OpenML no accesible. Cargando desde Drive...\n")
  read_csv("/content/drive/MyDrive/datasets/german_credit.csv", show_col_types = FALSE)
})

# Variables numéricas clave
num_vars <- c("duration", "credit_amount", "installment_commitment",
              "residence_since", "age", "existing_credits", "num_dependents")

cat(sprintf("\n  Dimensiones: %s filas × %d columnas\n",
            format(nrow(df_original), big.mark = ","), ncol(df_original)))
cat(sprintf("  Valores faltantes totales: %d\n", sum(is.na(df_original))))
cat(sprintf("\n  Estadísticos de credit_amount (ground truth):\n"))
cat(sprintf("    Media   = %.2f\n", mean(df_original$credit_amount)))
cat(sprintf("    Mediana = %.2f\n", median(df_original$credit_amount)))
cat(sprintf("    Desvío  = %.2f\n", sd(df_original$credit_amount)))


### 📝 Interpretación

El dataset está completo: 0 valores faltantes. La variable `credit_amount` exhibe la asimetría positiva típica de variables monetarias (media > mediana). Estos estadísticos constituyen nuestra **línea base** contra la cual evaluaremos cada estrategia de imputación.

---
## 3. Simulación de los tres mecanismos de Rubin

Introducimos un 20% de valores faltantes en `credit_amount` bajo cada mecanismo, de modo que podamos comparar cada imputación contra el valor verdadero.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 3. SIMULACIÓN DE MCAR, MAR, MNAR
# ══════════════════════════════════════════════════════════════

set.seed(42)
n <- nrow(df_original)
prop_miss <- 0.20

# ── 3.1 MCAR: ausencia puramente aleatoria ──
mask_mcar <- runif(n) < prop_miss

# ── 3.2 MAR: ausencia depende de 'age' (variable observada) ──
# Jóvenes tienen mayor probabilidad de no reportar el monto
prob_mar <- 1 / (1 + exp(0.08 * (df_original$age - 28)))
prob_mar <- prob_mar / mean(prob_mar) * prop_miss
prob_mar <- pmin(pmax(prob_mar, 0.01), 0.95)
mask_mar <- runif(n) < prob_mar

# ── 3.3 MNAR: ausencia depende del propio credit_amount ──
# Montos altos tienen mayor probabilidad de no ser reportados
ca <- df_original$credit_amount
prob_mnar <- 1 / (1 + exp(-0.0008 * (ca - quantile(ca, 0.7))))
prob_mnar <- prob_mnar / mean(prob_mnar) * prop_miss
prob_mnar <- pmin(pmax(prob_mnar, 0.01), 0.95)
mask_mnar <- runif(n) < prob_mnar

# Crear DataFrames con faltantes
df_mcar <- df_original; df_mcar$credit_amount[mask_mcar] <- NA
df_mar  <- df_original; df_mar$credit_amount[mask_mar]   <- NA
df_mnar <- df_original; df_mnar$credit_amount[mask_mnar] <- NA

cat("═══════════════════════════════════════════════════════════\n")
cat("RESUMEN DE FALTANTES SIMULADOS EN credit_amount\n")
cat("═══════════════════════════════════════════════════════════\n")

mecanismos <- list(
  list(nombre = "MCAR", mascara = mask_mcar),
  list(nombre = "MAR",  mascara = mask_mar),
  list(nombre = "MNAR", mascara = mask_mnar)
)

for (m in mecanismos) {
  mk <- m$mascara
  n_miss <- sum(mk)
  m_obs <- mean(df_original$credit_amount[!mk])
  m_mis <- mean(df_original$credit_amount[mk])
  dif <- abs(m_obs - m_mis)
  flag <- ifelse(dif > 200, "  ← sesgo!", "  ← sin sesgo")

  cat(sprintf("\n  %s:\n", m$nombre))
  cat(sprintf("    Faltantes: %d (%.1f%%)\n", n_miss, n_miss / n * 100))
  cat(sprintf("    Media de los OBSERVADOS:         %.0f\n", m_obs))
  cat(sprintf("    Media de los FALTANTES (verdad):  %.0f\n", m_mis))
  cat(sprintf("    Diferencia:                       %.0f%s\n", dif, flag))
}


### 📝 Interpretación

Este resultado constituye la **demostración empírica** de los teoremas del apunte teórico:

- **MCAR:** La media de los valores observados y la de los faltantes son prácticamente iguales. No hay sesgo: la ausencia es independiente del valor. Consecuencia: la *listwise deletion* produce estimaciones insesgadas (Teorema 3.1).
- **MAR:** Diferencia moderada. Los jóvenes (que solicitan montos distintos al promedio) tienen mayor probabilidad de faltante. Consecuencia: la *listwise deletion* introduce sesgo (Teorema 4.1).
- **MNAR:** Diferencia sustancial. Los montos altos faltan con mayor frecuencia, por lo que la media de los observados **subestima** la media real. Consecuencia: todos los métodos estándar están sesgados (Proposición 5.1).

---
## 4. Visualización de patrones de ausencia con `naniar`


In [ ]:
# ══════════════════════════════════════════════════════════════
# 4.1 VISUALIZACIÓN: vis_miss (patrón global)
# ══════════════════════════════════════════════════════════════

options(repr.plot.width = 14, repr.plot.height = 5)

# Creamos un DataFrame con faltantes en múltiples variables para visualizar
df_demo <- df_mar[num_vars]
# Agregamos faltantes en 'age' bajo MAR (depende de duration) para enriquecer
mask_age <- runif(n) < (0.15 * as.integer(df_original$duration > 24) + 0.05)
df_demo$age[mask_age] <- NA

p_vismiss <- vis_miss(df_demo, sort_miss = TRUE) +
  ggtitle("Patrón de ausencia (MAR simulado)") +
  theme(plot.title = element_text(face = "bold", size = 13))

print(p_vismiss)


In [ ]:
# ══════════════════════════════════════════════════════════════
# 4.2 FALTANTES POR VARIABLE
# ══════════════════════════════════════════════════════════════

options(repr.plot.width = 10, repr.plot.height = 5)

p_missvar <- gg_miss_var(df_demo) +
  ggtitle("Cantidad de valores faltantes por variable") +
  theme(plot.title = element_text(face = "bold", size = 13))

print(p_missvar)


In [ ]:
# ══════════════════════════════════════════════════════════════
# 4.3 SHADOW PLOT: distribución condicional a la ausencia
# ══════════════════════════════════════════════════════════════

# ¿La distribución de 'age' difiere entre quienes reportan credit_amount y quienes no?
options(repr.plot.width = 13, repr.plot.height = 5)

df_shadow <- df_mar %>%
  select(all_of(num_vars)) %>%
  bind_shadow() %>%
  mutate(credit_amount_NA = factor(
    credit_amount_NA,
    levels = c("!NA", "NA"),
    labels = c("Observado", "Faltante")
  ))

p_shadow <- ggplot(df_shadow, aes(x = age, fill = credit_amount_NA)) +
  geom_density(alpha = 0.6, color = "white", linewidth = 0.3) +
  scale_fill_manual(values = c(col_good, col_bad)) +
  labs(
    title = "Distribución de age según presencia/ausencia de credit_amount",
    subtitle = "Si las densidades difieren → la ausencia depende de age → mecanismo MAR",
    x = "Edad del solicitante", y = "Densidad",
    fill = "credit_amount"
  )

print(p_shadow)


### 📝 Interpretación

Las tres visualizaciones de `naniar` ofrecen información complementaria:

- **`vis_miss`** (patrón global): Las barras oscuras señalan datos presentes; las claras, faltantes. Si los faltantes se distribuyen uniformemente → MCAR. Si se concentran en ciertas filas → MAR o MNAR.
- **`gg_miss_var`** (faltantes por variable): Cuantifica la proporción de faltantes en cada variable. Permite priorizar el tratamiento.
- **Shadow plot (densidades):** La distribución de `age` difiere claramente entre quienes reportan `credit_amount` (verde) y quienes no (rojo). Los solicitantes jóvenes tienen mayor densidad en el grupo "Faltante". Esta diferencia visual confirma **MAR**: la ausencia depende de una variable observada (`age`). Si las densidades fueran idénticas, sería MCAR.

---
## 5. Diagnóstico del mecanismo de ausencia

### 5.1 Regresión logística del indicador de ausencia

Para cada mecanismo, modelamos $P(R_j = 0 \mid \mathbf{X})$ como función de las variables observadas. Un AUC significativamente > 0.5 indica que la ausencia es predecible → MAR o MNAR.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 5.1 REGRESIÓN LOGÍSTICA DEL INDICADOR DE AUSENCIA
# ══════════════════════════════════════════════════════════════

cat("═══════════════════════════════════════════════════════════\n")
cat("DIAGNÓSTICO: REGRESIÓN LOGÍSTICA DE R (indicador de ausencia)\n")
cat("═══════════════════════════════════════════════════════════\n")

# Variables predictoras para el diagnóstico (excluimos credit_amount)
X_diag <- df_original %>%
  select(duration, installment_commitment, residence_since,
         age, existing_credits, num_dependents)

for (m in mecanismos) {
  nm <- m$nombre; mk <- m$mascara

  # Crear el DataFrame para la regresión
  df_lr <- X_diag %>% mutate(R_missing = as.integer(mk))

  # Ajustar regresión logística
 fit <- glm(R_missing ~ ., data = df_lr, family = binomial)
  probs <- predict(fit, type = "response")

  # Calcular AUC-ROC con pROC
  roc_obj <- roc(df_lr$R_missing, probs, quiet = TRUE)
  auc_val <- as.numeric(auc(roc_obj))

  cat(sprintf("\n  %s:", nm))
  cat(sprintf("  AUC-ROC = %.4f", auc_val))

  if (auc_val < 0.55) {
    cat("  →  Ausencia NO predecible → compatible con MCAR ✅")
  } else if (auc_val < 0.65) {
    cat("  →  Predicción débil → posible MAR leve")
  } else {
    cat("  →  Ausencia PREDECIBLE → MAR confirmado ⚠️")
  }

  # Coeficientes más relevantes
  coefs <- abs(coef(fit)[-1])  # excluir intercept
  top_var <- names(sort(coefs, decreasing = TRUE))[1]
  cat(sprintf("\n    Variable más predictiva: '%s' (|β| = %.4f)\n", top_var, max(coefs)))
}


### 📝 Interpretación

- **MCAR:** AUC ≈ 0.50 — las variables observadas NO predicen la ausencia. La función `glm()` no encuentra ningún predictor significativo. Coherente con el mecanismo puramente aleatorio.
- **MAR:** AUC ≈ 0.67 — la variable `age` domina los coeficientes. La ausencia de `credit_amount` es predecible a partir de la edad del solicitante.
- **MNAR:** AUC moderado — las variables observadas capturan parte de la señal indirectamente (porque `credit_amount` correlaciona con `duration` y `age`), pero no toda, ya que la dependencia principal es con el valor faltante mismo.

### 5.2 Test de Little para MCAR


In [ ]:
# ══════════════════════════════════════════════════════════════
# 5.2 TEST DE LITTLE PARA MCAR
# ══════════════════════════════════════════════════════════════

cat("\n═══════════════════════════════════════════════════════════\n")
cat("TEST DE LITTLE (H₀: los datos son MCAR)\n")
cat("═══════════════════════════════════════════════════════════\n")

datasets_test <- list(
  list(nombre = "MCAR", datos = df_mcar),
  list(nombre = "MAR",  datos = df_mar),
  list(nombre = "MNAR", datos = df_mnar)
)

for (d in datasets_test) {
  tryCatch({
    test_result <- mcar_test(d$datos[num_vars])
    cat(sprintf("\n  %s:  χ² = %.2f,  df = %d,  p-valor = %.4f",
                d$nombre, test_result$statistic, test_result$df, test_result$p.value))
    if (test_result$p.value > 0.05) {
      cat("  →  No rechaza MCAR ✅")
    } else {
      cat("  →  RECHAZA MCAR → MAR o MNAR ⚠️")
    }
  }, error = function(e) {
    cat(sprintf("\n  %s:  Test no ejecutable (%s)", d$nombre, conditionMessage(e)))
  })
}
cat("\n")


### 📝 Interpretación

El test de Little evalúa la hipótesis nula $H_0$: los datos son MCAR. Si $p < 0.05$, se rechaza MCAR.

- Bajo **MCAR genuino**, el test NO rechaza (p > 0.05).
- Bajo **MAR y MNAR**, el test rechaza — pero no distingue entre ambos. Para diferenciar MAR de MNAR se requiere la regresión logística de la Sección 5.1 complementada con juicio experto del dominio.

### 5.3 Correlación point-biserial


In [ ]:
# ══════════════════════════════════════════════════════════════
# 5.3 CORRELACIÓN POINT-BISERIAL
# ══════════════════════════════════════════════════════════════

cat("\n═══════════════════════════════════════════════════════════\n")
cat("CORRELACIÓN POINT-BISERIAL: R(credit_amount) vs. variables\n")
cat("═══════════════════════════════════════════════════════════\n")

vars_corr <- c("duration", "age", "installment_commitment", "residence_since")

for (m in mecanismos) {
  cat(sprintf("\n  %s:\n", m$nombre))
  r_indicator <- as.integer(m$mascara)

  for (v in vars_corr) {
    test <- cor.test(r_indicator, df_original[[v]])
    r_val <- test$estimate
    p_val <- test$p.value
    sig <- ifelse(p_val < 0.001, "***",
           ifelse(p_val < 0.01, "**",
           ifelse(p_val < 0.05, "*", "ns")))
    cat(sprintf("    vs. %-28s r_pb = %+.4f  (p = %.4f) %s\n", v, r_val, p_val, sig))
  }
}


### 📝 Interpretación

- **MCAR:** Ninguna correlación significativa — confirma independencia total.
- **MAR:** Correlación fuerte y significativa (`***`) con `age` — la variable que condiciona la ausencia.
- **MNAR:** Correlaciones indirectas: `duration` y `age` capturan parte de la señal por estar correlacionadas con `credit_amount`.

---
## 6. Estrategias de imputación: implementación y comparación

Implementamos cinco estrategias de imputación y las evaluamos contra el *ground truth*.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 6. IMPUTACIÓN — 5 ESTRATEGIAS CON mice Y COMPARACIÓN
# ══════════════════════════════════════════════════════════════

# Función de evaluación contra ground truth
evaluate_imp <- function(mascara, verdaderos, imputados, metodo) {
  tv <- verdaderos[mascara]
  iv <- imputados[mascara]
  mae  <- mean(abs(tv - iv))
  rmse <- sqrt(mean((tv - iv)^2))
  sesgo <- mean(iv) - mean(tv)
  data.frame(
    Metodo     = metodo,
    MAE        = round(mae, 1),
    RMSE       = round(rmse, 1),
    Media_imp  = round(mean(iv), 1),
    Media_real = round(mean(tv), 1),
    Sesgo      = round(sesgo, 1),
    Sesgo_pct  = round(sesgo / mean(tv) * 100, 1),
    stringsAsFactors = FALSE
  )
}

cat("═══════════════════════════════════════════════════════════\n")
cat("COMPARACIÓN DE ESTRATEGIAS DE IMPUTACIÓN\n")
cat("═══════════════════════════════════════════════════════════\n")

true_ca <- df_original$credit_amount

datos_na <- list(
  list(nombre = "MCAR", df = df_mcar, mascara = mask_mcar),
  list(nombre = "MAR",  df = df_mar,  mascara = mask_mar),
  list(nombre = "MNAR", df = df_mnar, mascara = mask_mnar)
)

for (d in datos_na) {
  X_na <- d$df[num_vars]
  mk   <- d$mascara
  resultados <- list()

  # M1: Listwise deletion (media de los casos completos)
  mean_cc <- mean(X_na$credit_amount, na.rm = TRUE)
  imp_lw  <- X_na$credit_amount
  imp_lw[mk] <- mean_cc
  resultados[[1]] <- evaluate_imp(mk, true_ca, imp_lw, "Listwise (media CC)")

  # M2: Mediana
  med_val <- median(X_na$credit_amount, na.rm = TRUE)
  imp_med <- X_na$credit_amount
  imp_med[mk] <- med_val
  resultados[[2]] <- evaluate_imp(mk, true_ca, imp_med, "Mediana")

  # M3: mice — Predictive Mean Matching (PMM)
  imp_pmm <- mice(X_na, m = 1, method = "pmm", maxit = 5, printFlag = FALSE, seed = 42)
  X_pmm   <- complete(imp_pmm)
  resultados[[3]] <- evaluate_imp(mk, true_ca, X_pmm$credit_amount, "mice (PMM)")

  # M4: mice — CART (árboles de decisión)
  imp_cart <- mice(X_na, m = 1, method = "cart", maxit = 5, printFlag = FALSE, seed = 42)
  X_cart   <- complete(imp_cart)
  resultados[[4]] <- evaluate_imp(mk, true_ca, X_cart$credit_amount, "mice (CART)")

  # M5: mice — Regresión lineal con ruido (norm)
  imp_norm <- mice(X_na, m = 1, method = "norm", maxit = 5, printFlag = FALSE, seed = 42)
  X_norm   <- complete(imp_norm)
  resultados[[5]] <- evaluate_imp(mk, true_ca, X_norm$credit_amount, "mice (norm)")

  cat(sprintf("\n═══════════════════════════════════════════════════════════\n"))
  cat(sprintf("  MECANISMO: %s\n", d$nombre))
  cat(sprintf("═══════════════════════════════════════════════════════════\n\n"))
  res_df <- bind_rows(resultados)
  print(as.data.frame(res_df), row.names = FALSE)
}


### 📝 Interpretación

Los resultados confirman empíricamente los teoremas del apunte teórico:

**Bajo MCAR:**
- Todas las estrategias producen sesgo bajo. La listwise deletion es insesgada (Teorema 3.1).
- `mice` (PMM y CART) ofrecen menor MAE/RMSE porque aprovechan la estructura multivariada.

**Bajo MAR:**
- La listwise deletion y la mediana introducen **sesgo significativo**: la media de los *complete cases* difiere de la media real porque los jóvenes están subrepresentados (Teorema 4.1).
- **`mice` (PMM y CART) reducen sustancialmente el sesgo** porque condicionan la imputación en `age` y las demás predictoras — exactamente lo que predice la propiedad de ignorabilidad (Teorema 4.2).

**Bajo MNAR:**
- **Todos los métodos muestran sesgo residual** (Proposición 5.1). `mice` mejora respecto de la media/mediana, pero no elimina el sesgo porque la dependencia es con el valor faltante mismo — información inaccesible.
- La corrección bajo MNAR requiere modelos de selección (Heckman, 1979).

---
## 7. Visualización comparativa de distribuciones post-imputación


In [ ]:
# ══════════════════════════════════════════════════════════════
# 7. DISTRIBUCIONES POST-IMPUTACIÓN
# ══════════════════════════════════════════════════════════════

options(repr.plot.width = 17, repr.plot.height = 5.5)

plots_dist <- list()

for (d in datos_na) {
  X_na <- d$df[num_vars]

  # mice PMM
  imp <- mice(X_na, m = 1, method = "pmm", maxit = 5, printFlag = FALSE, seed = 42)
  X_imp <- complete(imp)

  # Media simple
  media_val <- mean(X_na$credit_amount, na.rm = TRUE)
  ca_media <- X_na$credit_amount
  ca_media[is.na(ca_media)] <- media_val

  # Combinar para ggplot
  df_plot <- bind_rows(
    tibble(credit_amount = df_original$credit_amount, Fuente = "Ground truth"),
    tibble(credit_amount = X_imp$credit_amount,       Fuente = "mice (PMM)"),
    tibble(credit_amount = ca_media,                   Fuente = "Media")
  ) %>%
  mutate(Fuente = factor(Fuente, levels = c("Ground truth", "mice (PMM)", "Media")))

  p <- ggplot(df_plot, aes(x = credit_amount, fill = Fuente)) +
    geom_density(alpha = 0.45, color = "white", linewidth = 0.3) +
    scale_fill_manual(values = c("Ground truth" = "gray50",
                                  "mice (PMM)"  = col_good,
                                  "Media"        = col_bad)) +
    labs(title = d$nombre, x = "credit_amount", y = "Densidad") +
    theme(legend.position = "bottom", legend.title = element_blank(),
          plot.title = element_text(hjust = 0.5))

  plots_dist[[d$nombre]] <- p
}

grid.arrange(
  grobs = plots_dist, ncol = 3,
  top = grid::textGrob(
    "Distribución post-imputación vs. ground truth por mecanismo",
    gp = grid::gpar(fontface = "bold", fontsize = 13)
  )
)


### 📝 Interpretación

Los gráficos de densidad superpuestos revelan visualmente el sesgo de cada estrategia:

- **MCAR:** Tanto `mice` (PMM) como la media producen distribuciones cercanas al *ground truth*. La imputación por media genera un leve pico artificial en el centro.
- **MAR:** La imputación por media produce un **spike** pronunciado que distorsiona la distribución. `mice` (PMM) preserva notablemente la forma original porque imputa con valores reales del vecindario predictivo condicionado en las variables observadas.
- **MNAR:** Ambas estrategias subestiman la cola derecha (montos altos), porque los valores faltantes provienen precisamente de esa zona de la distribución. `mice` mitiga parcialmente el sesgo pero no lo elimina.

---
## 8. Diagnóstico visual con `VIM`


In [ ]:
# ══════════════════════════════════════════════════════════════
# 8. VISUALIZACIÓN CON VIM: matrixplot y marginplot
# ══════════════════════════════════════════════════════════════

options(repr.plot.width = 13, repr.plot.height = 6)

# marginplot: distribución bivariada resaltando los faltantes
marginplot(df_mar[, c("credit_amount", "age")],
           col = c(col_good, col_bad, col_acc),
           main = "Marginplot: credit_amount vs. age (MAR)",
           pch = 19, cex = 0.6)


### 📝 Interpretación

El `marginplot` de VIM resulta una herramienta diagnóstica poderosa: en los márgenes del gráfico de dispersión, las barras rojas muestran la distribución de `age` para los registros donde `credit_amount` falta, y viceversa. Si las distribuciones marginales difieren entre presentes y faltantes, el mecanismo no es MCAR. En este caso, los faltantes de `credit_amount` se concentran entre los solicitantes más jóvenes — confirmación visual de MAR.

---
## 9. Resumen ejecutivo y buenas prácticas


In [ ]:
# ══════════════════════════════════════════════════════════════
# 9. RESUMEN EJECUTIVO
# ══════════════════════════════════════════════════════════════

cat("╔══════════════════════════════════════════════════════════════════╗\n")
cat("║               RESUMEN EJECUTIVO — VALORES FALTANTES (R)        ║\n")
cat("╠══════════════════════════════════════════════════════════════════╣\n")
cat("║                                                                ║\n")
cat("║  PROTOCOLO DIAGNÓSTICO:                                        ║\n")
cat("║  1. Cuantificar: naniar::vis_miss(), gg_miss_var()             ║\n")
cat("║  2. Test formal: naniar::mcar_test() [Little, 1988]            ║\n")
cat("║  3. Regresión:   glm(R ~ ., family=binomial) → AUC con pROC   ║\n")
cat("║  4. Visual:      VIM::marginplot(), naniar::bind_shadow()      ║\n")
cat("║                                                                ║\n")
cat("║  PROTOCOLO DE IMPUTACIÓN:                                      ║\n")
cat("║  1. mice(method='pmm')  → primera opción bajo MAR             ║\n")
cat("║  2. mice(method='cart') → alternativa para relaciones no       ║\n")
cat("║                           lineales                             ║\n")
cat("║  3. mice(method='norm') → si se asume normalidad multivariada  ║\n")
cat("║  4. NUNCA imputar en todo el dataset antes del split           ║\n")
cat("║                                                                ║\n")
cat("║  RESULTADO CLAVE (verificado empíricamente):                   ║\n")
cat("║  · MCAR → listwise válida (insesgada pero ineficiente)        ║\n")
cat("║  · MAR  → mice (PMM/CART) reducen sesgo significativamente     ║\n")
cat("║  · MNAR → todos sesgados → requiere modelos de selección       ║\n")
cat("║                                                                ║\n")
cat("║  PIPELINE EN R (buena práctica):                               ║\n")
cat("║  recipes::step_impute_knn() o step_impute_bag() dentro de      ║\n")
cat("║  un workflow de tidymodels → fit en train, bake en test        ║\n")
cat("╚══════════════════════════════════════════════════════════════════╝\n")


---
## 📚 Referencias

- **Van Buuren, S. & Groothuis-Oudshoorn, K.** (2011). mice: Multivariate Imputation by Chained Equations in R. *Journal of Statistical Software*, 45(3), 1–67.
- **Tierney, N. J. & Cook, D.** (2023). Expanding Tidy Data Principles to Facilitate Missing Data Exploration, Visualization and Assessment of Imputations. *JSS*, 105(7).
- **Kowarik, A. & Templ, M.** (2016). Imputation with the R Package VIM. *Journal of Statistical Software*, 74(7).
- **Little, R. J. A.** (1988). A Test of Missing Completely at Random. *JASA*, 83(404), 1198–1202.
- **Rubin, D. B.** (1976). Inference and Missing Data. *Biometrika*, 63(3), 581–592.
- **Díaz, D. E.** (2026). Apunte Clase 2, Parte I: *Valores Faltantes en Bases de Datos Financieras*. UTN FRRo.

---

> **Dr. Darío Ezequiel Díaz** · drdarioezequieldiaz@gmail.com
>
> Aplicaciones de Minería de Datos a Economía y Finanzas · Maestría en Minería de Datos · UTN FRRo · 2026
>
> *Notebook R correspondiente a la Clase 3 — Jueves 23 de abril de 2026*
